In [1]:
import os
import csv
import itertools
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt

# ---------------- Configuration ----------------
DATA_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\data\split_with_713"
OUTPUT_DIR = r"E:\Learning\UNSW\Term2\9444\group_project\outputs\plot\final\cnn9_base"
NUM_CLASSES = 39
BATCH_SIZE = 64
NUM_WORKERS = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 30
WEIGHT_DECAY = 5e-4
LR = 5e-4
PATIENCE = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True

# --------------- Data Transforms ---------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --------------- Dataloaders -------------------

def get_dataloaders(data_dir: str, batch_size: int, num_workers: int):
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_ds = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
    test_ds = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

# --------------- CNN‑9 Architecture ---------------
class CNN9(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256), nn.ReLU(), nn.Dropout(0.7),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN9(NUM_CLASSES).to(DEVICE)

# --------------- Optimizer & Scheduler ---------------
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler("cuda")

# --------------- Helper: Evaluate Predictions ---------------

def evaluate_predictions(model: nn.Module, loader: DataLoader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

# ------------------- Training Loop -------------------

def train():
    print(">> start:", flush=True)
    train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS)

    best_val_acc = 0.0
    wait = 0
    epochs, train_losses, val_accs, val_f1s = [], [], [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)

        y_true, y_pred = evaluate_predictions(model, val_loader)
        val_acc = accuracy_score(y_true, y_pred)
        val_f1 = f1_score(y_true, y_pred, average="macro")

        epochs.append(epoch)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        val_f1s.append(val_f1)

        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}", flush=True)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_cnn9_leaf.pth"))
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    # ---------------- Final Test Metrics ----------------
    y_true_test, y_pred_test = evaluate_predictions(model, test_loader)
    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_prec = precision_score(y_true_test, y_pred_test, average="macro")
    test_rec = recall_score(y_true_test, y_pred_test, average="macro")
    test_f1 = f1_score(y_true_test, y_pred_test, average="macro")

    print("\n===== FINAL TEST METRICS =====")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall   : {test_rec:.4f}")
    print(f"F1-Score : {test_f1:.4f}\n")

    # ---------------- Confusion Matrix ----------------
    cm = confusion_matrix(y_true_test, y_pred_test)
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix — Test Set")
    plt.colorbar(shrink=0.8)
    ticks = range(NUM_CLASSES)
    plt.xticks(ticks)
    plt.yticks(ticks)
    thresh = cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], "d"),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_test.png"), dpi=300)
    plt.close()

    # ---------------- Save Metrics to CSV ----------------
    with open(os.path.join(OUTPUT_DIR, "metrics.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_acc", "val_f1"])
        for e, l, a, f1_val in zip(epochs, train_losses, val_accs, val_f1s):
            writer.writerow([e, l, a, f1_val])

    # ---------------- Plot Curves ----------------
    # Training Loss
    plt.figure()
    plt.plot(epochs, train_losses, marker="o")
    plt.title("Training Loss vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"), dpi=300)
    plt.close()

    # Validation Accuracy
    plt.figure()
    plt.plot(epochs, val_accs, marker="o")
    plt.title("Validation Accuracy vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "validation_accuracy.png"), dpi=300)
    plt.close()

    # Validation F1
    plt.figure()
    plt.plot(epochs, val_f1s, marker="o")
    plt.title("Validation F1-Score vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("F1-Score")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "validation_f1.png"), dpi=300)
    plt.close()

if __name__ == "__main__":
    train()


>> start:
Epoch  1/30 | Loss: 3.0627 | Val Acc: 0.3594 | Val F1: 0.1910
Epoch  2/30 | Loss: 2.4872 | Val Acc: 0.5441 | Val F1: 0.4364
Epoch  3/30 | Loss: 2.1019 | Val Acc: 0.6760 | Val F1: 0.5792
Epoch  4/30 | Loss: 1.8196 | Val Acc: 0.7492 | Val F1: 0.6703
Epoch  5/30 | Loss: 1.5882 | Val Acc: 0.8128 | Val F1: 0.7592
Epoch  6/30 | Loss: 1.3975 | Val Acc: 0.8176 | Val F1: 0.7728
Epoch  7/30 | Loss: 1.2573 | Val Acc: 0.8755 | Val F1: 0.8482
Epoch  8/30 | Loss: 1.1673 | Val Acc: 0.8471 | Val F1: 0.8143
Epoch  9/30 | Loss: 1.0869 | Val Acc: 0.8906 | Val F1: 0.8646
Epoch 10/30 | Loss: 1.0207 | Val Acc: 0.8590 | Val F1: 0.8322
Epoch 11/30 | Loss: 0.9617 | Val Acc: 0.8904 | Val F1: 0.8687
Epoch 12/30 | Loss: 0.8947 | Val Acc: 0.9078 | Val F1: 0.8875
Epoch 13/30 | Loss: 0.8608 | Val Acc: 0.9204 | Val F1: 0.9016
Epoch 14/30 | Loss: 0.8233 | Val Acc: 0.9216 | Val F1: 0.9053
Epoch 15/30 | Loss: 0.7810 | Val Acc: 0.9321 | Val F1: 0.9181
Epoch 16/30 | Loss: 0.7442 | Val Acc: 0.9179 | Val F1: 0.902